# CND Metrics Plotting

Notebook for `CndStatisticsRecorder` outputs:
- `BilevelCND_*_*_quality_time.csv`
- `BilevelCND_*_*_metadata.json`
- `BilevelCND_run_summary.csv`

In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')

In [ ]:
def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / 'performance_results').exists():
        return cwd
    if (cwd.parent / 'performance_results').exists():
        return cwd.parent
    raise FileNotFoundError('Cannot find performance_results directory from current notebook path')

PROJECT_ROOT = resolve_project_root()
PERFORMANCE_ROOT = PROJECT_ROOT / 'performance_results'
PROJECT_ROOT, PERFORMANCE_ROOT

In [ ]:
TRACE_PATTERN = re.compile(r'^BilevelCND_(?P<approach>.+)_(?P<run_id>\d+)_quality_time\.csv$')

def load_cnd_dataset(dataset_name: str):
    dataset_dir = PERFORMANCE_ROOT / dataset_name
    if not dataset_dir.exists():
        raise FileNotFoundError(f'Dataset directory not found: {dataset_dir}')

    trace_frames = []
    metadata_rows = []

    for trace_file in sorted(dataset_dir.glob('BilevelCND_*_quality_time.csv')):
        m = TRACE_PATTERN.match(trace_file.name)
        if not m:
            continue

        approach = m.group('approach')
        run_id = m.group('run_id')
        df = pd.read_csv(trace_file)

        if 'RunId' not in df.columns:
            df['RunId'] = run_id
        if 'Approach' not in df.columns:
            df['Approach'] = approach

        df['RunLabel'] = df['Approach'].astype(str) + ' | run=' + df['RunId'].astype(str)
        df['TraceFile'] = trace_file.name
        trace_frames.append(df)

        metadata_file = dataset_dir / f'BilevelCND_{approach}_{run_id}_metadata.json'
        if metadata_file.exists():
            with open(metadata_file, 'r', encoding='utf-8') as f:
                meta = json.load(f)
            metadata_rows.append(meta)

    trace_df = pd.concat(trace_frames, ignore_index=True) if trace_frames else pd.DataFrame()
    metadata_df = pd.DataFrame(metadata_rows)

    summary_file = dataset_dir / 'BilevelCND_run_summary.csv'
    summary_df = pd.read_csv(summary_file) if summary_file.exists() else pd.DataFrame()

    return trace_df, metadata_df, summary_df


In [ ]:
DATASET = 'SiouxFalls'
trace_df, metadata_df, summary_df = load_cnd_dataset(DATASET)

print('dataset:', DATASET)
print('trace rows:', len(trace_df))
print('runs:', trace_df['RunId'].nunique() if not trace_df.empty else 0)
trace_df.head(3)

In [ ]:
if trace_df.empty:
    print('No CND trace files found for this dataset')
else:
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))

    for run_label, g in trace_df.sort_values('ElapsedTime(s)').groupby('RunLabel'):
        axes[0].plot(g['ElapsedTime(s)'], g['BestFeasibleObjective'], label=run_label)
        axes[1].plot(g['ElapsedTime(s)'], g['BudgetViolation'], label=run_label)
        axes[2].plot(g['ElapsedTime(s)'], g['TAComputeTime(s)'], label=run_label)

    axes[0].set_title('Best feasible objective vs time')
    axes[0].set_xlabel('Elapsed time (s)')
    axes[0].set_ylabel('Best feasible objective')

    axes[1].set_title('Budget violation vs time')
    axes[1].set_xlabel('Elapsed time (s)')
    axes[1].set_ylabel('Budget violation')

    axes[2].set_title('TA compute time vs time')
    axes[2].set_xlabel('Elapsed time (s)')
    axes[2].set_ylabel('TA compute time (s)')

    for ax in axes:
        ax.legend(loc='best', fontsize=8)

    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 5))
    sns.boxplot(data=trace_df, x='Phase', y='TAComputeTime(s)', hue='Approach')
    plt.title('TA compute time distribution by phase')
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()

In [ ]:
if not metadata_df.empty:
    display_cols = [
        'run_id', 'timestamp', 'dataset', 'approach', 'algorithm',
        'design_variables', 'number_of_links', 'number_of_od_pairs',
        'max_standard_iterations', 'max_optimality_condition_iterations',
        'optimization_tolerance', 'budget_upper_bound'
    ]
    print('metadata rows:', len(metadata_df))
    display(metadata_df[[c for c in display_cols if c in metadata_df.columns]])
else:
    print('No metadata JSON files found')

if not summary_df.empty:
    print('summary rows:', len(summary_df))
    display(summary_df.tail(10))
else:
    print('No run summary CSV found')